In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

RAW_PATH = '../data/raw/application_train.csv'
df = pd.read_csv(RAW_PATH)
print('Original shape:', df.shape)

Original shape: (307511, 122)


In [15]:
# ── Use 40% of data (up from 15%) for better model quality ──
SAMPLE_FRAC = 0.40
df_sampled, _ = train_test_split(df, train_size=SAMPLE_FRAC, random_state=42, stratify=df['TARGET'])
print('After sampling shape:', df_sampled.shape)
df = df_sampled.copy()

After sampling shape: (123004, 122)


In [16]:
# ── Cleaning ──
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# ── Richer feature engineering ──
df['CREDIT_INCOME_RATIO']    = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO']   = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['EMPLOYED_AGE_RATIO']     = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['CREDIT_DURATION_RATIO']  = df['AMT_CREDIT'] / df['AMT_ANNUITY'].replace(0, np.nan)
df['GOODS_CREDIT_RATIO']     = df['AMT_GOODS_PRICE'] / df['AMT_CREDIT'].replace(0, np.nan)
df['AGE_YEARS']              = (-df['DAYS_BIRTH'] / 365).round(1)
df['YEARS_EMPLOYED']         = (-df['DAYS_EMPLOYED'] / 365).round(1)
df['INCOME_PER_PERSON']      = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].replace(0, np.nan)
df['DOC_COUNT']              = df.filter(like='FLAG_DOCUMENT_').sum(axis=1)
print('Feature engineering done. Shape:', df.shape)

Feature engineering done. Shape: (123004, 131)


In [17]:
# ── Drop high-missing + ID ──
drop_cols = ['SK_ID_CURR'] + df.columns[df.isnull().mean() > 0.45].tolist()
df = df.drop(columns=drop_cols)
print('Shape after dropping:', df.shape)

X = df.drop('TARGET', axis=1)
y = df['TARGET']

num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')

Shape after dropping: (123004, 81)
Numeric: 68, Categorical: 12


C:\Users\sharo\AppData\Local\Temp\ipykernel_18612\1763417156.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


In [18]:
# ── Save column lists for predict_single.py ──
import json
with open('../models/feature_columns.json', 'w') as f:
    json.dump({'num_cols': num_cols, 'cat_cols': cat_cols, 'all_cols': num_cols + cat_cols}, f, indent=2)
print('Column lists saved to ../models/feature_columns.json')

Column lists saved to ../models/feature_columns.json


In [19]:
# ── Preprocessing pipeline ──
num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
], remainder='drop')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print('X_train shape:', X_train.shape)

X_train shape: (98403, 80)


In [20]:
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)
print('Processed shapes:', X_train_proc.shape, X_test_proc.shape)

# Save with column names intact (not just numbered)
X_train_df = pd.DataFrame(X_train_proc)
X_test_df  = pd.DataFrame(X_test_proc)
X_train_df.to_csv('../data/processed/X_train.csv', index=False)
X_test_df.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save raw (unprocessed) splits for predict_single.py to use column names
X_train.to_csv('../data/processed/X_train_raw.csv', index=False)
X_test.to_csv('../data/processed/X_test_raw.csv', index=False)

joblib.dump(preprocessor, '../models/preprocessor.pkl')
print('Preprocessor saved.')

Processed shapes: (98403, 192) (24601, 192)
Preprocessor saved.
